# 5회차 · 강화학습 기초 — 상과 벌로 배우는 AI 만들기

**2026 경기도 공유학교 · 인공지능 기초 (김순찬 교수)** · 성취기준 [인일 03-03]

## 오늘의 학습목표
1. 강화학습의 구성 요소(에이전트·환경·상태·행동·보상)를 미로 게임에서 찾아 말할 수 있다.
2. Q-learning이 "점수표(Q-table)"를 고쳐 가며 배우는 원리를 설명할 수 있다.
3. **보상 체계를 직접 설계**하고, 보상이 달라지면 AI의 행동이 어떻게 달라지는지 실험으로 보인다.

## 사용법
- 셀을 **위에서부터 순서대로** 실행하세요. 셀을 클릭하고 `Shift + Enter`를 누르면 실행됩니다.
- `🧩 직접 해보기` 셀은 `# TODO` 줄을 여러분이 고쳐서 실행합니다. 막히면 바로 아래 **정답 보기**를 펼치세요.
- 오류가 나면 당황하지 말고 **런타임 → 런타임 다시 시작** 후 처음 셀부터 다시 실행하면 대부분 해결됩니다.

## 0. 준비 — 도구 불러오기와 한글 폰트
`numpy`는 숫자 계산, `matplotlib`는 그림, `pandas`는 표를 담당합니다. 그래프에 한글이 깨지지 않도록 나눔 글꼴을 설치합니다(Colab에서 10초쯤 걸립니다). 이 셀은 실행 결과가 거의 안 보여도 정상입니다.

In [ ]:
import matplotlib.pyplot as plt, matplotlib as mpl, numpy as np, pandas as pd
import subprocess, sys
try:
    subprocess.run(['apt-get','-qq','install','-y','fonts-nanum'], check=False, capture_output=True)   # Colab 한글 폰트
    import matplotlib.font_manager as fm; fm.fontManager.addfont('/usr/share/fonts/truetype/nanum/NanumGothic.ttf'); plt.rcParams['font.family']='NanumGothic'
except Exception:
    plt.rcParams['font.family'] = 'Noto Sans CJK JP'   # 로컬 검증용
plt.rcParams['axes.unicode_minus'] = False
print('준비 완료! numpy', np.__version__)

## 1. 환경 만들기 — 5×5 미로 (GridWorld)
강화학습에는 **환경(environment)** 이 필요합니다. 오늘의 환경은 5×5 칸짜리 미로입니다.

| 기호 | 뜻 | 위치 |
|---|---|---|
| **S** | 시작(Start) | (0, 0) 왼쪽 위 |
| **G** | 목표(Goal) — 도착하면 게임 끝 | (4, 4) 오른쪽 아래 |
| **X** | 함정(Trap) — 빠지면 게임 끝 | (1, 3), (2, 1), (3, 3) |

- **상태(state)** = 에이전트가 지금 있는 칸의 좌표 `(행, 열)`
- **행동(action)** = 상(0)·하(1)·좌(2)·우(3) 네 가지. 벽 쪽으로 가면 제자리에 머뭅니다.
- **보상(reward)** = 목표에 도착하면 `r_goal`, 함정에 빠지면 `r_trap`, 한 걸음마다 `r_step`. **이 세 숫자가 오늘 여러분이 설계할 "보상표"입니다.**

아래 코드는 환경 클래스와, 미로를 그려 주는 함수입니다. 내용을 다 이해하지 못해도 괜찮습니다 — `reset()`은 시작 위치로 되돌리고, `step(행동)`은 한 걸음 움직인 뒤 **(새 위치, 보상, 끝났는지)** 를 돌려준다는 것만 기억하세요.

In [ ]:
class GridWorld:
    def __init__(self, n=5, start=(0, 0), goal=(4, 4), traps=((1, 3), (2, 1), (3, 3)), walls=(),
                 r_goal=10.0, r_trap=0.0, r_step=0.0, max_steps=50):
        self.n = n; self.start = start; self.goal = goal
        self.traps = set(traps); self.walls = set(walls)
        self.r_goal, self.r_trap, self.r_step, self.max_steps = r_goal, r_trap, r_step, max_steps
        self.actions = [(-1, 0), (1, 0), (0, -1), (0, 1)]   # 0=상, 1=하, 2=좌, 3=우  (행 변화, 열 변화)
        self.arrows = ['↑', '↓', '←', '→']
        self.reset()

    def reset(self):                       # 새 게임 시작: 시작 칸으로
        self.pos = self.start; self.t = 0
        return self.pos

    def step(self, a):                     # 행동 a를 한 번 실행
        dr, dc = self.actions[a]
        r, c = self.pos[0] + dr, self.pos[1] + dc
        if 0 <= r < self.n and 0 <= c < self.n and (r, c) not in self.walls:
            self.pos = (r, c)              # 미로 안이고 벽이 아니면 이동
        self.t += 1
        reward = self.r_step; done = False # 한 걸음 비용(0이면 없음)
        if self.pos == self.goal:  reward += self.r_goal; done = True   # 목표 도착
        elif self.pos in self.traps: reward += self.r_trap; done = True # 함정
        if self.t >= self.max_steps: done = True                        # 너무 오래 헤매면 강제 종료
        return self.pos, reward, done


def draw_grid(ax, env, Q=None, path=None, title=None):
    """미로 그리기. Q를 주면 각 칸의 최선 행동을 화살표로, path를 주면 지나간 길을 주황 선으로."""
    n = env.n; img = np.ones((n, n, 3))
    for (r, c) in env.traps: img[r, c] = (0.98, 0.80, 0.82)
    for (r, c) in env.walls: img[r, c] = (0.55, 0.55, 0.55)
    img[env.goal] = (0.80, 0.94, 0.85); img[env.start] = (0.85, 0.88, 0.98)
    ax.imshow(img, extent=(-0.5, n - 0.5, n - 0.5, -0.5))
    ax.set_xticks(np.arange(-0.5, n, 1)); ax.set_yticks(np.arange(-0.5, n, 1))
    ax.set_xticklabels([]); ax.set_yticklabels([]); ax.grid(True, color='#9CA3AF'); ax.tick_params(length=0)
    ax.text(env.start[1], env.start[0], 'S', ha='center', va='center', fontsize=14, fontweight='bold', color='#1B2A4A')
    ax.text(env.goal[1], env.goal[0], 'G', ha='center', va='center', fontsize=14, fontweight='bold', color='#0E8C86')
    for (r, c) in env.traps: ax.text(c, r, 'X', ha='center', va='center', fontsize=14, fontweight='bold', color='#C0395F')
    if Q is not None:
        for r in range(n):
            for c in range(n):
                if (r, c) in env.traps or (r, c) == env.goal or (r, c) in env.walls or np.all(Q[r, c] == 0): continue
                dr, dc = env.actions[int(np.argmax(Q[r, c]))]
                ax.annotate('', xy=(c + dc * 0.32, r + dr * 0.32), xytext=(c - dc * 0.15, r - dr * 0.15),
                            arrowprops=dict(arrowstyle='->', lw=2, color='#4F5BD5'))
    if path:
        ax.plot([p[1] for p in path], [p[0] for p in path], '-', color='#D97706', lw=3, alpha=.7)
    if title: ax.set_title(title)

env = GridWorld()
fig, ax = plt.subplots(figsize=(4, 4)); draw_grid(ax, env, title='우리의 미로 (S 시작, G 목표, X 함정)'); plt.show()
print('시작 위치:', env.reset(), '/ 목표:', env.goal, '/ 함정:', sorted(env.traps))

## 2. 아무렇게나 움직이는 에이전트 — 왜 "학습"이 필요할까?
먼저 **배우지 않은 에이전트**를 봅시다. 매번 상·하·좌·우를 주사위 던지듯 고릅니다.
`env.step(a)`가 돌려주는 세 값을 한 줄씩 출력해서 **상태 → 행동 → 보상**의 흐름을 눈으로 확인하세요. (같은 결과가 나오도록 난수 씨앗 `seed=2`를 고정했습니다.)

In [ ]:
rng = np.random.default_rng(2)             # 난수 생성기 (씨앗 고정 → 매번 같은 결과)
env = GridWorld(r_goal=10, r_trap=-10, r_step=0)
s = env.reset(); path = [s]; done = False; total = 0
print('걸음 | 행동 | 새 위치 | 보상 | 끝?')
while not done:
    a = int(rng.integers(4))               # 0~3 중 아무 행동
    s, r, done = env.step(a)               # 한 걸음
    path.append(s); total += r
    print(f'{env.t:>3}  | {env.arrows[a]}    | {s}  | {r:+.0f}   | {done}')
print('총 보상 =', total, '/ 도착:', s == env.goal, '/ 함정:', s in env.traps)
fig, ax = plt.subplots(figsize=(4, 4)); draw_grid(ax, env, path=path, title=f'무작위 에이전트의 길 ({len(path)-1}걸음)'); plt.show()

### 무작위 에이전트는 100판 중 몇 판이나 성공할까?
한 판만 보면 운이 좋았는지 나빴는지 알 수 없습니다. **100판**을 돌려 성공률을 재 봅시다. 이 숫자가 오늘 우리가 이길 "기준선"입니다.

In [ ]:
def play_random(env, n_games=100, seed=0):
    rng = np.random.default_rng(seed); wins = 0; steps = []
    for g in range(n_games):
        s = env.reset(); done = False
        while not done:
            s, r, done = env.step(int(rng.integers(4)))
        wins += (s == env.goal); steps.append(env.t)
    return wins / n_games, np.mean(steps)

win_rate, avg_steps = play_random(GridWorld(r_goal=10, r_trap=-10))
print(f'무작위 에이전트: 성공률 {win_rate:.0%}, 평균 {avg_steps:.1f}걸음 만에 게임 종료')

## 3. Q-table — 에이전트의 "점수표" 만들기
Q-learning의 핵심은 **Q-table**입니다. 칸(상태)마다 네 가지 행동 각각에 대해 **"이 칸에서 이 행동을 하면 앞으로 총 얼마나 좋을까"** 를 숫자로 적어 둔 표입니다.

- 크기: 5행 × 5열 × 4행동 = **100개의 숫자**. 처음엔 아무것도 모르니 전부 0.
- 학습 = 경험할 때마다 이 숫자를 조금씩 고치는 것.
- 다 배우고 나면, 각 칸에서 **가장 큰 값의 행동**을 고르면 됩니다.

In [ ]:
Q = np.zeros((5, 5, 4))                    # (행, 열, 행동) → 점수
print('Q-table 크기:', Q.shape, '→ 숫자 개수:', Q.size)
print('시작 칸 (0,0)의 네 행동 점수 [↑ ↓ ← →]:', Q[0, 0])
print('아직 아무것도 배우지 않아서 전부 0 입니다.')

## 4. 행동 고르기 — 탐험(explore) vs 활용(exploit)
점수표에서 **가장 큰 값의 행동만** 고르면(활용) 처음 우연히 찾은 길만 고집하게 됩니다. 그래서 **ε(입실론)의 확률로는 일부러 아무 행동이나** 해 봅니다(탐험).

- `eps = 0.2` → 10번 중 2번은 모험, 8번은 아는 대로.
- 아래 함수는 이 규칙(**ε-greedy**)을 그대로 코드로 옮긴 것입니다. 점수가 같은 행동이 여럿이면 그중 하나를 무작위로 고릅니다(전부 0인 처음에 특히 중요).

In [ ]:
def choose_action(Q, s, eps, rng):
    if rng.random() < eps:                 # eps 확률로 탐험: 아무 행동
        return int(rng.integers(4))
    q = Q[s]                               # 활용: 점수가 가장 큰 행동 (동점이면 무작위)
    best = np.flatnonzero(q == q.max())
    return int(rng.choice(best))

rng = np.random.default_rng(0)
Q_test = np.zeros((5, 5, 4)); Q_test[0, 0] = [0, 2, 0, 5]    # (0,0)에서 → 가 5점으로 최고라고 가정
picks = [choose_action(Q_test, (0, 0), eps=0.2, rng=rng) for _ in range(1000)]
counts = np.bincount(picks, minlength=4)
print('1000번 고른 결과 [↑ ↓ ← →]:', counts, '→ 대부분 →(3) 이지만 가끔 다른 행동도 시도')

## 5. Q-learning 갱신 규칙 — 점수표를 어떻게 고칠까?
한 걸음 움직여 `(상태 s, 행동 a, 보상 r, 새 상태 s')`를 경험할 때마다 다음 한 줄로 점수를 고칩니다.

$$Q(s,a) \leftarrow Q(s,a) + \alpha\,\big[\, r + \gamma \max_{a'} Q(s',a') - Q(s,a) \,\big]$$

말로 풀면: **"지금 적힌 점수"를 "방금 받은 보상 + 다음 칸에서 기대되는 최고 점수"쪽으로 조금(α만큼) 옮긴다.**

| 기호 | 이름 | 뜻 | 오늘 값 |
|---|---|---|---|
| α (알파) | 학습률 | 한 번에 얼마나 크게 고칠까 (0~1) | 0.1 |
| γ (감마) | 할인율 | 미래 보상을 얼마나 중요하게 볼까 (1에 가까울수록 멀리 봄) | 0.9 |
| ε (입실론) | 탐험 확률 | 얼마나 자주 모험할까 | 0.2 |

게임이 끝나는 칸(목표·함정)에서는 "다음 칸의 기대 점수"가 없으므로 보상 r만 씁니다. 이 규칙을 여러 판(에피소드) 반복하는 함수가 `q_learning`입니다.

In [ ]:
def q_learning(env, episodes=1500, alpha=0.1, gamma=0.9, eps=0.2, seed=0):
    rng = np.random.default_rng(seed)
    Q = np.zeros((env.n, env.n, 4)); history = []
    for ep in range(episodes):
        s = env.reset(); done = False; total = 0.0
        while not done:
            a = choose_action(Q, s, eps, rng)              # ① 행동 고르기 (ε-greedy)
            s2, r, done = env.step(a)                      # ② 환경에서 한 걸음
            ended = done and (s2 == env.goal or s2 in env.traps)
            target = r if ended else r + gamma * Q[s2].max()   # ③ 목표값: 보상 + 미래 기대치
            Q[s][a] += alpha * (target - Q[s][a])          # ④ 점수표 갱신 (핵심 한 줄!)
            s = s2; total += r
        history.append(total)                              # 이 판의 총 보상 기록
    return Q, np.array(history)

def greedy_path(env, Q, limit=30):
    """배운 점수표대로만(탐험 없이) 움직였을 때의 길"""
    s = env.reset(); path = [s]; done = False
    while not done and len(path) < limit:
        s, r, done = env.step(int(np.argmax(Q[s]))); path.append(s)
    return path

print('함수 준비 완료')

## 6. 학습시키기 — 1500판을 하고 나면?
보상표는 **목표 +10, 함정 −10, 한 걸음 −0.1** 로 두고 1500판을 학습시킵니다. 몇 초면 끝납니다.
학습 곡선에서 볼 것: **처음(왼쪽)** 에는 함정에 자주 빠져 총 보상이 −10 근처, **뒤로 갈수록** +10 근처로 올라가야 합니다. 회색 선은 판마다 들쭉날쭉(탐험 때문에)하니 파란 **이동 평균**을 보세요.

In [ ]:
env = GridWorld(r_goal=10, r_trap=-10, r_step=-0.1)
Q, hist = q_learning(env, episodes=1500, alpha=0.1, gamma=0.9, eps=0.2, seed=0)

fig, ax = plt.subplots(figsize=(8, 3.5))
ax.plot(hist, color='#D1D5DB', lw=.8, label='판마다 총 보상')
ax.plot(pd.Series(hist).rolling(50).mean(), color='#4F5BD5', lw=2.5, label='50판 이동 평균')
ax.set_xlabel('에피소드(판)'); ax.set_ylabel('총 보상'); ax.legend(); ax.set_title('학습 곡선'); plt.show()
print(f'처음 100판 평균 보상: {hist[:100].mean():+.2f}   마지막 100판 평균 보상: {hist[-100:].mean():+.2f}')

## 7. 배운 정책 보기 — 각 칸의 화살표
**정책(policy)** = "어느 칸에서 무엇을 할지"의 규칙. 점수표에서 칸마다 최고 점수 행동을 화살표로 그리면 정책이 보입니다. 주황 선은 탐험 없이 화살표만 따라갔을 때의 길입니다.

확인할 것: 화살표가 함정을 **피해서** 목표로 흐르는가? 시작에서 목표까지 최단 거리는 8걸음입니다(오른쪽 4 + 아래 4).

In [ ]:
path = greedy_path(env, Q)
fig, ax = plt.subplots(figsize=(4.5, 4.5)); draw_grid(ax, env, Q=Q, path=path, title='학습된 정책과 길'); plt.show()
print('길:', path)
print(f'걸음 수: {len(path)-1}, 도착 여부: {path[-1] == env.goal}')
print('시작 칸 (0,0)의 점수 [↑ ↓ ← →]:', np.round(Q[0, 0], 2))

## 8. 보상 체계를 바꾸면 행동이 바뀐다 — 4가지 설계 비교 ★
알고리즘은 그대로 두고 **보상표만** 바꿔 봅니다. 이것이 오늘 수업의 핵심 실험입니다.

| 설계 | 목표 | 함정 | 한 걸음 | 예상 |
|---|---|---|---|---|
| ① 목표만 | +10 | 0 | 0 | 함정에 벌은 없지만 게임이 끝나므로 피하긴 함 |
| ② 함정 벌점 | +10 | −10 | 0 | 함정을 더 확실히 피함(안전한 길 선호) |
| ③ 걸음 비용 | +10 | −10 | −0.1 | 오래 헤매면 손해 → 짧은 길 선호 |
| ④ 잘못된 설계 | +10 | **+5** | 0 | 함정에 상을 주면? → **보상 해킹** |

각 설계로 학습한 뒤 정책 그림과 표를 비교하세요. 표의 `마지막100판평균`은 학습이 잘 됐는지, `걸음수`·`결과`는 실제 행동을 보여 줍니다.

In [ ]:
designs = [
    ('① 목표만 +10',      dict(r_goal=10, r_trap=0,   r_step=0)),
    ('② 함정 -10 추가',    dict(r_goal=10, r_trap=-10, r_step=0)),
    ('③ 걸음마다 -0.1',    dict(r_goal=10, r_trap=-10, r_step=-0.1)),
    ('④ 함정 +5 (실수)',   dict(r_goal=10, r_trap=5,   r_step=0)),
]
rows = []
fig, axes = plt.subplots(1, 4, figsize=(15, 4))
for ax, (name, kw) in zip(axes, designs):
    e = GridWorld(**kw)
    Qd, hd = q_learning(e, episodes=1500, seed=0)
    p = greedy_path(e, Qd)
    result = '도착' if p[-1] == e.goal else ('함정!' if p[-1] in e.traps else '실패')
    rows.append({'설계': name, **kw, '걸음수': len(p) - 1, '결과': result, '마지막100판평균': round(hd[-100:].mean(), 2)})
    draw_grid(ax, e, Q=Qd, path=p, title=f'{name}\n{len(p)-1}걸음 · {result}')
plt.show()
results = pd.DataFrame(rows); results

### 표를 읽어 봅시다
- ①②③은 모두 **8걸음(최단)** 으로 도착하지만 **지나가는 길이 다릅니다**. 이 미로에는 8걸음짜리 길이 여러 개라서, 어느 길을 고르는지는 보상표와 탐험 중 우연히 먼저 발견한 길에 따라 달라집니다. 함정 벌점(②)을 주면 함정 옆을 덜 지나는 벽 쪽 길을 고르는 경향이 보입니다.
- `마지막100판평균`이 ①>②>③ 순으로 낮아지는 것은 실력이 나빠서가 아니라 **보상표 자체가 달라서**입니다(②는 탐험 중 함정에 빠지면 −10, ③은 걸음마다 −0.1이 빠짐). 서로 다른 보상표의 점수는 직접 비교하면 안 됩니다 — 비교할 것은 **행동(길)** 입니다.
- ④는 **함정에 상(+5)** 을 줬더니 에이전트가 목표 대신 가장 가까운 함정으로 직행합니다. 잘못 설계한 보상은 AI가 "규칙의 허점"을 파고들게 만듭니다. 이것을 **보상 해킹(reward hacking)** 이라고 합니다.
- 결론: **AI가 무엇을 배우느냐는 알고리즘이 아니라 보상표가 정합니다.** 그래서 보상 설계가 중요합니다.

## 🧩 직접 해보기 1 — 함정 위치를 바꿔 보자
함정을 **최단 경로 위**에 놓으면 에이전트는 어떻게 돌아갈까요? `# TODO` 줄의 함정 좌표를 바꾸고 실행하세요. (좌표는 `(행, 열)`, 0~4. 시작 (0,0)과 목표 (4,4)는 피하세요.)

In [ ]:
my_traps = ((1, 3), (2, 1), (3, 3))        # TODO: 함정 좌표를 바꿔 보세요. 예: ((0, 2), (2, 2), (4, 2))
e = GridWorld(traps=my_traps, r_goal=10, r_trap=-10, r_step=-0.1)
Qm, hm = q_learning(e, episodes=1500, seed=0); pm = greedy_path(e, Qm)
fig, ax = plt.subplots(figsize=(4.5, 4.5)); draw_grid(ax, e, Q=Qm, path=pm, title=f'내 미로: {len(pm)-1}걸음'); plt.show()
print('도착:', pm[-1] == e.goal, '/ 마지막 100판 평균 보상:', round(hm[-100:].mean(), 2))

<details><summary><b>정답 보기 (예시)</b></summary>

`my_traps = ((0, 2), (2, 2), (4, 2))` 처럼 가운데 세로줄을 막으면 에이전트는 (1,2) 또는 (3,2)를 지나 돌아갑니다. 벽처럼 완전히 막으면(`(0,2),(1,2),(2,2),(3,2),(4,2)`) 길이 없어 학습 곡선이 오르지 않습니다 — "환경에 답이 없으면 아무리 학습해도 못 찾는다"는 것도 중요한 관찰입니다.
</details>

## 🧩 직접 해보기 2 — 탐험 확률 ε을 바꾸면?
`eps_list`에 값을 바꿔 넣고, 어느 ε에서 학습이 가장 빠르고 안정적인지 학습 곡선을 비교하세요. ε=0(탐험 없음)과 ε=0.5(절반은 모험)도 꼭 넣어 보세요.

In [ ]:
eps_list = [0.0, 0.05, 0.2, 0.5]           # TODO: 값을 바꿔 보세요
env = GridWorld(r_goal=10, r_trap=-10, r_step=-0.1)
fig, ax = plt.subplots(figsize=(8, 3.5))
for eps in eps_list:
    Qe, he = q_learning(env, episodes=1500, eps=eps, seed=0)
    ax.plot(pd.Series(he).rolling(50).mean(), lw=2, label=f'ε = {eps}  (마지막100판 {he[-100:].mean():+.1f})')
ax.set_xlabel('에피소드'); ax.set_ylabel('총 보상 (50판 이동 평균)'); ax.legend(); ax.set_title('탐험 확률에 따른 학습 곡선'); plt.show()

<details><summary><b>정답 보기 · 해설</b></summary>

- **ε = 0**: 처음 우연히 찾은 길에 갇히기 쉽습니다. 운이 좋으면 빨리 배우지만 더 좋은 길을 못 찾을 수 있습니다.
- **ε = 0.05~0.2**: 보통 가장 안정적입니다.
- **ε = 0.5**: 배운 뒤에도 절반은 모험하므로 **학습 중 총 보상**은 낮게 보입니다(함정에 자주 빠짐). 하지만 점수표 자체는 잘 배워서, 탐험을 끄고 `greedy_path`로 걸으면 잘 도착합니다. "학습 중 점수"와 "배운 뒤 실력"은 다르다는 점을 기억하세요.
</details>

## (선택) 9. 진짜 라이브러리로 — gymnasium FrozenLake
연구자들이 쓰는 강화학습 환경 모음 **gymnasium**에도 우리 미로와 똑같은 구조의 게임 `FrozenLake`가 있습니다(4×4 얼음판, H=구멍, G=목표). 상태가 좌표 대신 0~15 번호인 것만 다르고, 우리가 만든 Q-learning을 거의 그대로 쓸 수 있습니다. 미끄러지지 않는 버전(`is_slippery=False`)으로 짧게 해 봅니다.

In [ ]:
try:
    import gymnasium as gym
    fl = gym.make('FrozenLake-v1', is_slippery=False)
    nS, nA = fl.observation_space.n, fl.action_space.n
    Qf = np.zeros((nS, nA)); rng = np.random.default_rng(0); wins = []
    for ep in range(2000):
        s, _ = fl.reset(seed=int(rng.integers(1_000_000))); done = False
        while not done:
            a = int(rng.integers(nA)) if rng.random() < 0.2 else int(rng.choice(np.flatnonzero(Qf[s] == Qf[s].max())))
            s2, r, term, trunc, _ = fl.step(a); done = term or trunc
            Qf[s, a] += 0.1 * ((r if term else r + 0.9 * Qf[s2].max()) - Qf[s, a]); s = s2
        wins.append(r)
    print(f'FrozenLake 4x4: 상태 {nS}개, 행동 {nA}개 / 마지막 200판 성공률 {np.mean(wins[-200:]):.0%}')
    arrows = np.array(['←', '↓', '→', '↑'])[Qf.argmax(1)].reshape(4, 4)   # FrozenLake 행동 번호: 0좌 1하 2우 3상
    print('배운 정책 (H=구멍 자리 포함):'); print('\n'.join(' '.join(row) for row in arrows))
except ImportError:
    print('gymnasium이 없습니다. Colab에서는 셀 맨 위에  !pip install gymnasium  을 추가한 뒤 다시 실행하세요.')

## 📝 오늘의 결과물 제출 — 나의 보상표 설계
아래 셀에서 **이름·학번**과 **나만의 보상표**(목표·함정·걸음 값)를 정하고, 왜 그렇게 정했는지 한두 문장을 씁니다. 실행하면 학습 결과와 정책 그림이 나옵니다. 그림과 출력 내용을 캡처해 제출하세요.

평가 기준: ① 보상표 세 값이 의도에 맞게 설계되었는가 ② 결과 정책이 의도대로 나왔는가(안 나왔다면 그 이유를 설명했는가) ③ 설명에 상태·행동·보상 용어를 바르게 썼는가.

In [ ]:
이름 = '홍길동'            # TODO
학번 = '10101'             # TODO
나의_보상표 = dict(r_goal=10, r_trap=-10, r_step=-0.1)    # TODO: 세 값을 직접 설계
설계_이유 = '함정은 크게 벌주고, 오래 헤매면 조금씩 손해를 줘서 안전하고 짧은 길을 배우게 하려고.'   # TODO

e = GridWorld(**나의_보상표)
Qs, hs = q_learning(e, episodes=1500, seed=0); ps = greedy_path(e, Qs)
result = '도착' if ps[-1] == e.goal else ('함정!' if ps[-1] in e.traps else '실패')
fig, ax = plt.subplots(figsize=(4.5, 4.5)); draw_grid(ax, e, Q=Qs, path=ps, title=f'{이름}의 보상표 → {len(ps)-1}걸음 · {result}'); plt.show()
print('=' * 50); print(f'{이름} ({학번}) · 5회차 강화학습 결과물'); print('보상표:', 나의_보상표)
print('설계 이유:', 설계_이유)
print(f'결과: {result}, {len(ps)-1}걸음 / 마지막 100판 평균 보상 {hs[-100:].mean():+.2f} / 처음 100판 {hs[:100].mean():+.2f}')
print('길:', ps); print('=' * 50)

## ✅ 자기 점검 체크리스트
- [ ] 미로 게임에서 **에이전트·환경·상태·행동·보상**이 각각 무엇인지 말할 수 있다.
- [ ] Q-table이 무엇을 저장하는 표인지, 크기가 왜 5×5×4인지 설명할 수 있다.
- [ ] ε-greedy에서 ε이 커지면/작아지면 어떻게 되는지 안다.
- [ ] Q-learning 갱신 한 줄을 말로 풀어 설명할 수 있다.
- [ ] 보상표를 바꿨을 때 정책이 어떻게 달라졌는지 예를 들어 말할 수 있다(보상 해킹 포함).
- [ ] 나의 보상표 설계와 결과를 제출했다.

**다음 시간 예고**: 5회차로 인공지능 기초 과정이 마무리됩니다. 오늘 배운 "보상으로 배우는 AI"는 알파고, 로봇 걷기, ChatGPT의 답변 다듬기(RLHF)에도 쓰이는 원리입니다.